# Loading Reconstructions and trying to clean

## Before

In [1]:
import open3d as o3d
pcd = o3d.io.read_point_cloud("../reconstructions/vggt_10_False.ply")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
try:
    o3d.visualization.draw_geometries([pcd], mesh_show_back_face=True)
except Exception as e:
    print(f"Visualizer error: {e}")

## After

In [3]:
import sys
sys.path.append("..")
from utils.pointCloud import clean_point_cloud, remove_outliers

### Radius Method

In [ ]:
pcd_cleaned_radius = clean_point_cloud(pcd, method='radius', radius=0.5, min_points=30)
o3d.visualization.draw_geometries([pcd_cleaned_radius], mesh_show_back_face=True)

### Statistical method

In [7]:
pcd_cleaned_stats = clean_point_cloud(pcd, method='statistical', nb_neighbors=50, std_ratio=1.0)
o3d.visualization.draw_geometries([pcd_cleaned_stats], mesh_show_back_face=True)

### Cluster

In [4]:
pcd_cleaned_cluster = remove_outliers(pcd, eps=0.02, min_points=10)
o3d.visualization.draw_geometries([pcd_cleaned_cluster], mesh_show_back_face=True)

## Mesh Generation

In [6]:
import open3d as o3d
import numpy as np

# Read the point cloud
pcd = o3d.io.read_point_cloud("../reconstructions/colmap_20_False.ply")

# Estimate normals (required for surface reconstruction)
pcd.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30)
)

# Orient normals consistently
pcd.orient_normals_consistent_tangent_plane(k=15)

# Surface Reconstruction (often better quality)
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
    pcd, depth=15
)
vertices_to_remove = densities < np.quantile(densities, 0.01)
mesh.remove_vertices_by_mask(vertices_to_remove)

# Save the mesh
o3d.io.write_triangle_mesh("../reconstructions/mesh_colmap.ply", mesh)

# Visualize the result
o3d.visualization.draw_geometries([mesh])

[Open3D WARNING] Write Ply clamped color value to valid range
[Open3D WARNING] GLFW Error: WGL: Failed to make context current: The requested transformation operation is not supported. 


It is better to convert to Mesh, using only the vertices removal as cleaning